# 01 — EDA

Display only. All logic lives in `src/s6e7/` (CLAUDE.md rule 5).
Every plot below is implemented by hand in `src/s6e7/plots.py`; cells raise
`NotImplementedError` until the corresponding function is written.

Build order is `PLOTS_SPEC.md`: `target_overview` and `missingness` first.

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from s6e7 import eda, io, plots

plt.rcParams["figure.dpi"] = 120
pl.Config.set_tbl_rows(30)

polars.config.Config

In [2]:
train = io.load_train()
test = io.load_test()

train.shape, test.shape

((690088, 15), (295753, 14))

## Step 1 — column overview

**Decision:** which columns are numeric vs categorical, which need imputation, and
which are effectively constant.

`n_unique` here excludes nulls. Polars' own `n_unique` counts null as a distinct value,
which reports a 3-level categorical with missing data as having 4 levels.

In [3]:
eda.overview(train)

column,dtype,nulls,null_pct,n_unique
str,str,i64,f64,i64
"""id""","""UInt32""",0,0.0,690088
"""health_condition""","""String""",0,0.0,3
"""sleep_duration""","""Float32""",75999,11.01,701
"""heart_rate""","""Float32""",7833,1.14,537
"""bmi""","""Float32""",13898,2.01,1596
"""calorie_expenditure""","""Float32""",52853,7.66,2101
"""step_count""","""Float32""",13916,2.02,12807
"""exercise_duration""","""Float32""",6901,1.0,856
"""water_intake""","""Float32""",43477,6.3,400


In [4]:
eda.overview(test)

column,dtype,nulls,null_pct,n_unique
str,str,i64,f64,i64
"""id""","""UInt32""",0,0.0,295753
"""sleep_duration""","""Float32""",32571,11.01,692
"""heart_rate""","""Float32""",3357,1.14,526
"""bmi""","""Float32""",5956,2.01,1548
"""calorie_expenditure""","""Float32""",22652,7.66,2068
"""step_count""","""Float32""",5964,2.02,12196
"""exercise_duration""","""Float32""",2958,1.0,817
"""water_intake""","""Float32""",18633,6.3,392
"""diet_type""","""String""",2958,1.0,3


## Step 2 — category levels

**Decision:** encoding strategy, and whether naive encoders are safe.

A level appearing only in **test** has no encoding learned for it and breaks a fitted
encoder at predict time. A level only in **train** is dead weight.

Levels print alphabetically, which is rarely the meaningful order — `high, low, medium`
sorts nothing like `low < medium < high`. Reading which of these are genuinely *ordinal*
is the judgement this step exists to support.

In [5]:
eda.category_levels(train, io.CATEGORICAL_COLS, test)

column,n_levels,levels,test_only,train_only
str,i64,str,str,str
"""diet_type""",3,"""balanced, non-veg, veg""","""""",""""""
"""stress_level""",3,"""high, low, medium""","""""",""""""
"""sleep_quality""",3,"""average, good, poor""","""""",""""""
"""physical_activity_level""",3,"""active, moderate, sedentary""","""""",""""""
"""smoking_alcohol""",3,"""no, occasional, yes""","""""",""""""
"""gender""",3,"""female, male, other""","""""",""""""


## Step 3 — class rate per level

**Decision:** is the target *monotone* in the declared level ordering?

That is the empirical claim an ordinal integer encoding makes, and the precondition for
a monotone constraint later. Read each column's block top to bottom: if a class rate
moves consistently in one direction, the ordering is real. If it zigzags, a threshold
split cannot isolate the middle level, and one-hot — or native categorical handling —
wins.

Orderings come from `io.ORDINAL_LEVELS`, a **modelling judgement, not a fact**. Edit it
there if you disagree.

Nulls appear as their own level, so this doubles as the missingness test for the
categoricals. Compare every row against the global rates: `at-risk` 0.859,
`unhealthy` 0.084, `fit` 0.058.

In [6]:
eda.level_target_rates(train, io.CATEGORICAL_COLS, io.TARGET, io.ORDINAL_LEVELS)

column,level,rows,share_pct,p_at-risk,p_fit,p_unhealthy
str,str,i64,f64,f64,f64,f64
"""diet_type""","""balanced""",226888,32.88,0.8514,0.0612,0.0874
"""diet_type""","""non-veg""",224867,32.59,0.8678,0.0516,0.0806
"""diet_type""","""veg""",231432,33.54,0.8567,0.0604,0.083
"""diet_type""","""<null>""",6901,1.0,0.8676,0.0509,0.0816
"""stress_level""","""low""",167708,24.3,0.7967,0.2006,0.0028
"""stress_level""","""medium""",261819,37.94,0.9939,0.003,0.0031
"""stress_level""","""high""",177750,25.76,0.7178,0.0035,0.2787
"""stress_level""","""<null>""",82811,12.0,0.8589,0.0575,0.0836
"""sleep_quality""","""poor""",212166,30.74,0.8296,0.0346,0.1358


## Step 4 — numeric summary

**Decision:** which features need transformation, whether a bound is a real limit or a
clip, and whether the distribution hides a point mass.

`grid` is the **median** gap between adjacent distinct values — the typical quantisation
step. Not the minimum: a single off-grid value splits one normal step into two smaller
ones and drags the minimum down tenfold, making a clean grid look ragged. `min_gap` sits
in its own column so contamination stays visible instead of corrupting `grid`.

`pct_at_min` / `pct_at_max` expose clipping. A genuine distribution *tapers* at its
extremes; a clipped one *piles up* there — percent-at-bound in whole numbers rather than
hundredths.

`mode_pct` catches a point mass **anywhere** — zero-inflation, a sentinel, a default —
which the bound percentages miss unless the spike happens to land on a boundary.

Note that transformations are moot for a GBDT regardless of what `skew` says: a tree
split is a question about rank order, and every monotone transform preserves rank order.
This table matters for clipping and point masses, and for any linear model later in the
blend.

In [7]:
eda.numeric_summary(train, io.NUMERIC_COLS)

column,n_unique,min,max,grid,min_gap,mode,mode_pct,pct_at_min,pct_at_max,mean,std,skew
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""sleep_duration""",701,3.0,10.0,0.01,0.009999,7.16,0.56,0.01,0.03,6.993,1.215,-0.008
"""heart_rate""",537,50.0,107.699997,0.099998,0.029999,74.0,0.58,0.01,0.0,75.097,8.175,0.0
"""bmi""",1596,16.0,34.82,0.01,0.000999,23.440001,0.33,0.03,0.0,22.985,2.482,0.025
"""calorie_expenditure""",2101,1200.0,3580.0,1.0,1.0,2201.0,0.41,0.0,0.0,2226.085,347.532,-0.184
"""step_count""",12807,1002.0,14999.0,1.0,1.0,12182.0,0.12,0.0,0.0,8615.953,3929.4,-0.18
"""exercise_duration""",856,0.0,99.800003,0.1,0.02,0.0,2.41,2.41,0.0,38.751,14.742,-0.375
"""water_intake""",400,0.5,4.72,0.01,0.01,2.37,1.04,0.05,0.0,2.189,0.518,0.11


## Step 5 — is missingness a feature?

**Decision:** add missing-indicator columns, or don't bother.

If the class rate differs between missing and present, missingness carries information
and deserves an indicator. If the rates match, the nulls were injected at random and an
indicator is dead weight — and clever imputation buys nothing either.

Sorted by `abs_diff`, largest first. The `<null>` rows in step 3 already hint at the
answer for the categoricals.

In [8]:
eda.missing_vs_target(train, io.FEATURE_COLS, io.TARGET).sort("abs_diff", descending=True).head(12)

column,target_class,n_missing,p_when_missing,p_when_present,abs_diff
str,str,i64,f64,f64,f64
"""bmi""","""unhealthy""",13898,0.0293,0.0848,0.0555
"""bmi""","""at-risk""",13898,0.8888,0.8581,0.0307
"""bmi""","""fit""",13898,0.082,0.0572,0.0248
"""heart_rate""","""at-risk""",7833,0.8708,0.8585,0.0123
"""heart_rate""","""unhealthy""",7833,0.0739,0.0838,0.0098
"""diet_type""","""at-risk""",6901,0.8676,0.8586,0.009
"""exercise_duration""","""at-risk""",6901,0.8509,0.8588,0.0079
"""diet_type""","""fit""",6901,0.0509,0.0577,0.0069
"""exercise_duration""","""unhealthy""",6901,0.0887,0.0836,0.0051


In [9]:
eda.missing_cooccurrence(train, io.FEATURE_COLS).head(10)

col_a,col_b,both_missing,expected,ratio
str,str,i64,f64,f64
"""diet_type""","""gender""",630,213.7,2.95
"""heart_rate""","""gender""",267,242.6,1.1
"""bmi""","""physical_activity_level""",811,737.5,1.1
"""diet_type""","""smoking_alcohol""",313,285.8,1.1
"""bmi""","""diet_type""",150,139.0,1.08
"""exercise_duration""","""gender""",228,213.7,1.07
"""sleep_duration""","""diet_type""",795,760.0,1.05
"""bmi""","""sleep_quality""",1221,1174.8,1.04
"""exercise_duration""","""diet_type""",72,69.0,1.04


## Step 6 — which numeric features carry signal

**Decision:** which features to plot, and what to expect from a baseline.

`spread_sd` is the gap between the largest and smallest class mean, in units of the
column's own standard deviation — a standardised effect size. Dividing is what makes
steps comparable to hours.

`overlap_pct` and `best_split_acc` restate the same number in units you can act on.
Modelling each class as a normal with equal variance and means `d` apart, the shared
area is `2 * Phi(-d/2)`, and the best accuracy a **single threshold** can reach is
`Phi(d/2)`. So `d = 2.1` means one cut separates the extreme classes ~86% of the time;
`d = 0.05` is a coin flip.

Build density plots **only** for the features that score here.

Three blind spots. Dividing by the *overall* std understates separation, so these are
conservative. It compares means only — equal means with different variances score zero
and are still separable, which is exactly where a plot beats a table. And with three
classes it reads only the extremes: a feature that splits one class off while leaving the
other two superimposed scores the same as one that separates all three. **Read the
per-class means, not just the ranking.**

In [10]:
eda.class_profile(train, io.NUMERIC_COLS, io.TARGET)

column,mean_at-risk,mean_fit,mean_unhealthy,overall_std,spread_sd,overlap_pct,best_split_acc
str,f64,f64,f64,f64,f64,f64,f64
"""sleep_duration""",7.086,7.954,5.366,1.215,2.1295,28.7,0.857
"""bmi""",22.95,21.829,24.123,2.482,0.9244,64.4,0.678
"""step_count""",8406.706,11651.307,8670.234,3929.4,0.8257,68.0,0.66
"""exercise_duration""",37.965,50.042,39.044,14.742,0.8192,68.2,0.659
"""calorie_expenditure""",2214.943,2363.992,2245.42,347.532,0.4289,83.0,0.585
"""heart_rate""",75.101,74.804,75.257,8.175,0.0554,97.8,0.511
"""water_intake""",2.189,2.179,2.189,0.518,0.0192,99.2,0.504


## Step 7 — redundancy between numerics

**Decision:** which features are duplicates of each other.

Two columns correlated above ~0.95 are one feature wearing two hats: the second adds no
information, splits the importance between them so neither looks strong, and
destabilises any linear model in the blend.

Spearman by default — rank-based, so it catches monotone-but-curved relationships that
Pearson underestimates, and outliers don't move it. Nulls drop pairwise, so `n_pairs`
shows how many rows each figure actually rests on.

A GBDT tolerates redundancy far better than a linear model. Read this mainly as a
warning about split importance, and as a pruning list for the day a linear model or a
network joins the blend.

In [11]:
eda.numeric_correlation(train, io.NUMERIC_COLS)

col_a,col_b,n_pairs,spearman,abs
str,str,i64,f64,f64
"""step_count""","""exercise_duration""",669402,0.4413,0.4413
"""calorie_expenditure""","""exercise_duration""",630836,0.3698,0.3698
"""calorie_expenditure""","""step_count""",624390,0.3667,0.3667
"""bmi""","""calorie_expenditure""",624389,0.1098,0.1098
"""sleep_duration""","""bmi""",601658,-0.0613,0.0613
"""bmi""","""step_count""",662540,-0.0176,0.0176
"""bmi""","""exercise_duration""",669419,-0.0174,0.0174
"""heart_rate""","""exercise_duration""",675423,-0.007,0.007
"""heart_rate""","""step_count""",668491,-0.0064,0.0064
